In [1]:
import sys
import pypyodbc as odbc
from PyQt5 import QtWidgets
from student_information import Ui_MainWindow  # your generated UI file


# ------------------------ DATABASE CONFIG ------------------------
DRIVER_NAME = "ODBC Driver 18 for SQL Server"
SERVER_NAME = "localhost"
DATABASE_NAME = "University"
USERNAME = "SA"
PASSWORD = "Lenashalewis!"

connection_string = f"""
    DRIVER={{{DRIVER_NAME}}};
    SERVER={SERVER_NAME};
    DATABASE={DATABASE_NAME};
    UID={USERNAME};
    PWD={PASSWORD};
    TrustServerCertificate=yes;
"""

def connect_db():
    return odbc.connect(connection_string)


# ------------------------ MAIN APPLICATION ------------------------
class StudentApp(QtWidgets.QMainWindow):
    def __init__(self):
        super().__init__()
        self.ui = Ui_MainWindow()
        self.ui.setupUi(self)

        self.records = []
        self.current_index = -1

        # Button Connections
        self.ui.pushButton.clicked.connect(self.add_record)      # ADD
        self.ui.pushButton_6.clicked.connect(self.update_record) # UPDATE
        self.ui.pushButton_4.clicked.connect(self.delete_record) # DELETE
        self.ui.pushButton_5.clicked.connect(self.clear_fields)  # CANCEL
        self.ui.pushButton_7.clicked.connect(self.reset_db)      # RESET
        self.ui.pushButton_2.clicked.connect(self.next_record)   # NEXT
        self.ui.pushButton_3.clicked.connect(self.prev_record)   # PREV

        # SEARCH using text field (press Enter)
        self.ui.lineEdit_9.returnPressed.connect(self.search_record)

        self.load_records()
        if self.records:
            self.display_record(self.records[0])

    # ------------------------ LOAD RECORDS ------------------------
    def load_records(self):
        conn = connect_db()
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM Students")
        self.records = cursor.fetchall()
        self.current_index = 0 if self.records else -1

    # ------------------------ DISPLAY RECORD ------------------------
    def display_record(self, row):
        # RegistrationNumber, FirstName, SecondName, Surname, Course, County, Gender, Department, School
        self.ui.lineEdit.setText(row[0])   # Reg No
        self.ui.lineEdit_2.setText(row[1]) # First Name
        self.ui.lineEdit_3.setText(row[2]) # Second Name
        self.ui.lineEdit_4.setText(row[3]) # Surname
        self.ui.lineEdit_7.setText(row[4]) # Course
        self.ui.lineEdit_6.setText(row[5]) # County
        self.ui.lineEdit_5.setText(row[7]) # Department
        self.ui.lineEdit_8.setText(row[8]) # School

        # Gender
        gender = row[6].strip() 
        if gender == "Male":
            self.ui.radioButton.setChecked(True)
            self.ui.radioButton_2.setChecked(False)
            self.ui.radioButton_3.setChecked(False)
        elif gender == "Female":
            self.ui.radioButton_2.setChecked(True)
            self.ui.radioButton.setChecked(False)
            self.ui.radioButton_3.setChecked(False)
        else:
            self.ui.radioButton_3.setChecked(True)
            self.ui.radioButton.setChecked(False)
            self.ui.radioButton_2.setChecked(False)

    # ------------------------ GET GENDER ------------------------
    def get_gender(self):
        if self.ui.radioButton.isChecked():
            return "Male"
        elif self.ui.radioButton_2.isChecked():
            return "Female"
        else:
            return "Other"

    # ------------------------ VALIDATION ------------------------
    def validate(self):
        if not self.ui.lineEdit.text():
            QtWidgets.QMessageBox.warning(self, "Error", "Registration Number is required")
            return False
        if not self.ui.lineEdit_2.text():
            QtWidgets.QMessageBox.warning(self, "Error", "First Name is required")
            return False
        if not self.ui.lineEdit_3.text():
            QtWidgets.QMessageBox.warning(self, "Error", "Second Name is required")
            return False
        if not self.ui.lineEdit_4.text():
            QtWidgets.QMessageBox.warning(self, "Error", "Surname is required")
            return False
        return True

    # ------------------------ ADD RECORD ------------------------
    def add_record(self):
        if not self.validate():
            return
        try:
            conn = connect_db()
            cursor = conn.cursor()
            cursor.execute("""
                INSERT INTO Students (RegistrationNumber, FirstName, SecondName, Surname, Course, County, Gender, Department, School)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                self.ui.lineEdit.text(),
                self.ui.lineEdit_2.text(),
                self.ui.lineEdit_3.text(),
                self.ui.lineEdit_4.text(),
                self.ui.lineEdit_7.text(),
                self.ui.lineEdit_6.text(),
                self.get_gender(),
                self.ui.lineEdit_5.text(),
                self.ui.lineEdit_8.text()
            ))
            conn.commit()
            QtWidgets.QMessageBox.information(self, "Success", "Record Added")
            self.load_records()
        except Exception as e:
            QtWidgets.QMessageBox.warning(self, "Error", str(e))

    # ------------------------ UPDATE RECORD ------------------------
    def update_record(self):
        if not self.validate():
            return
        try:
            conn = connect_db()
            cursor = conn.cursor()
            cursor.execute("""
                UPDATE Students SET
                FirstName=?, SecondName=?, Surname=?, Course=?, County=?, Gender=?, Department=?, School=?
                WHERE RegistrationNumber=?
            """, (
                self.ui.lineEdit_2.text(),
                self.ui.lineEdit_3.text(),
                self.ui.lineEdit_4.text(),
                self.ui.lineEdit_7.text(),
                self.ui.lineEdit_6.text(),
                self.get_gender(),
                self.ui.lineEdit_5.text(),
                self.ui.lineEdit_8.text(),
                self.ui.lineEdit.text()
            ))
            conn.commit()
            QtWidgets.QMessageBox.information(self, "Updated", "Record Updated")
            self.load_records()
        except Exception as e:
            QtWidgets.QMessageBox.warning(self, "Error", str(e))

    # ------------------------ DELETE ------------------------
    def delete_record(self):
        reply = QtWidgets.QMessageBox.question(
            self, "Confirm", "Delete this record?",
            QtWidgets.QMessageBox.Yes | QtWidgets.QMessageBox.No
        )
        if reply == QtWidgets.QMessageBox.Yes:
            try:
                conn = connect_db()
                cursor = conn.cursor()
                cursor.execute(
                    "DELETE FROM Students WHERE RegistrationNumber=?",
                    self.ui.lineEdit.text()
                )
                conn.commit()
                QtWidgets.QMessageBox.warning(self, "Deleted", "Record Deleted")
                self.load_records()
            except Exception as e:
                QtWidgets.QMessageBox.warning(self, "Error", str(e))

    # ------------------------ SEARCH ------------------------
    def search_record(self):
        try:
            conn = connect_db()
            cursor = conn.cursor()
            search_value = self.ui.lineEdit_9.text()
            cursor.execute("""
                SELECT * FROM Students 
                WHERE RegistrationNumber=? 
                OR (FirstName + ' ' + SecondName)=?
            """, (search_value, search_value))
            row = cursor.fetchone()
            if row:
                self.display_record(row)
                QtWidgets.QMessageBox.information(self, "Found", "Record Found")
            else:
                QtWidgets.QMessageBox.warning(self, "Not Found", "No record found")
        except Exception as e:
            QtWidgets.QMessageBox.warning(self, "Error", str(e))

    # ------------------------ NEXT ------------------------
    def next_record(self):
        if self.records and self.current_index < len(self.records) - 1:
            self.current_index += 1
            self.display_record(self.records[self.current_index])

    # ------------------------ PREVIOUS ------------------------
    def prev_record(self):
        if self.records and self.current_index > 0:
            self.current_index -= 1
            self.display_record(self.records[self.current_index])

    # ------------------------ CLEAR FIELDS ------------------------
    def clear_fields(self):
        self.ui.lineEdit.clear()
        self.ui.lineEdit_2.clear()
        self.ui.lineEdit_3.clear()
        self.ui.lineEdit_4.clear()
        self.ui.lineEdit_5.clear()
        self.ui.lineEdit_6.clear()
        self.ui.lineEdit_7.clear()
        self.ui.lineEdit_8.clear()
        self.ui.lineEdit_9.clear()
        self.ui.radioButton.setChecked(False)
        self.ui.radioButton_2.setChecked(False)
        self.ui.radioButton_3.setChecked(False)

    # ------------------------ RESET DATABASE ------------------------
    def reset_db(self):
        reply = QtWidgets.QMessageBox.question(
            self, "Confirm", "Delete ALL records?",
            QtWidgets.QMessageBox.Yes | QtWidgets.QMessageBox.No
        )
        if reply == QtWidgets.QMessageBox.Yes:
            try:
                conn = connect_db()
                cursor = conn.cursor()
                cursor.execute("DELETE FROM Students")
                conn.commit()
                QtWidgets.QMessageBox.warning(self, "Reset", "Database Cleared")
                self.load_records()
            except Exception as e:
                QtWidgets.QMessageBox.warning(self, "Error", str(e))


# ------------------------ RUN APPLICATION ------------------------
if __name__ == "__main__":
    app = QtWidgets.QApplication(sys.argv)
    window = StudentApp()
    window.show()
    sys.exit(app.exec_())

SystemExit: 0

/home/lennox-lewis/jupyter-env/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
